#Import Libraries & Data

In [ ]:
#import libraries
import pandas as pd
import pandas_gbq
import pydata_google_auth
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

import matplotlib.style as style
style.available
style.use('fivethirtyeight')

##Authenticate

In [ ]:
import pandas_gbq
import pydata_google_auth

SCOPES = [
    'https://www.googleapis.com/auth/cloud-platform',
    'https://www.googleapis.com/auth/drive',
]

# get credentials
credentials = pydata_google_auth.get_user_credentials(
    SCOPES,
    auth_local_webserver=False)

In [ ]:
# GBQ
sql = """
select *
# from `ffn-dw-bigquery-prd.Dexter.v_aal_early_pay_default`
from `ffam-data-platform-loan-ops.report_views.v_aal_early_pay_default`
where extract(year from loan_vintage_month) >= 2026
"""

# run query
df = pandas_gbq.read_gbq(sql
                         ,project_id='ffn-dw-bigquery-prd'
                         ,credentials=credentials
                         ,dialect='standard'
                        )

In [ ]:
df.head()

In [ ]:
# Get high-level statistics of the DataFrame
display(df.describe().T)

In [ ]:
display(df.mode().T)

In [ ]:
# Calculate and display null counts per column
null_counts = df.isnull().sum()
print("Null counts per column:")
display(null_counts)

#Feature Engineering
##Identify Important features impacting `flag_1st_pay_default`

### Subtask:
Load the data, preprocess it, perform feature selection, train a classification model, and identify the most important features influencing `flag_1st_pay_default`.

**Reasoning**:
To identify the most important features, I will perform the following steps:
1. Load the data from BigQuery.
2. Handle missing values by dropping columns with all nulls and then imputing the remaining ones.
3. Encode categorical features.
4. Select features using a method like SelectKBest.
5. Train a classification model (e.g., RandomForestClassifier) on the preprocessed and selected data.
6. Extract and display the feature importances from the trained model.

In [ ]:
# Data Preprocessing
# Separate target variable
X = df.drop('flag_1st_pay_default', axis=1)
y = df['flag_1st_pay_default']

# Drop columns with all missing values
X.dropna(axis=1, how='all', inplace=True)

# Identify categorical and numerical columns after dropping
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=np.number).columns

# Impute missing values (using mean for numerical and most frequent for categorical)
numerical_imputer = SimpleImputer(strategy='mean')
X[numerical_cols] = numerical_imputer.fit_transform(X[numerical_cols])

categorical_imputer = SimpleImputer(strategy='most_frequent')
X[categorical_cols] = categorical_imputer.fit_transform(X[categorical_cols])


# Encode categorical features
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Drop date and dbdate columns before feature selection
date_cols = X.select_dtypes(include=['datetime64[ns]', 'dbdate']).columns
X = X.drop(columns=date_cols)

# Check for constant columns before feature selection
constant_cols = [col for col in X.columns if X[col].nunique() == 1]
print(f"Constant columns before feature selection: {constant_cols}")

# Check for remaining NaNs before feature selection
print("Number of NaNs per column before feature selection:")
display(X.isnull().sum())

# Define the list of categorical features to explicitly include
categorical_features_to_include = [
    'loan_origination_loan_amount_band'
    , 'loan_origination_interest_rate_band'
    , 'loan_origination_principal_period'
    , 'borrower_language'
    , 'borrower_fico_bands_origination'
    , 'borrower_annual_income_band'
    , 'borrower_origination_risk_group_twentile'
    , 'borrower_origination_state'
    , 'loan_origination_first_payment_day_of_week'
    , 'days_elapsed_bucket_negotiation'
    , 'loan_application_NDI_ratio'
    , 'loan_application_PTI_ratio'
    , 'loan_application_utm_lead_source'
    , 'loan_application_utm_lead_channel'
    , 'loan_origination_interest_rate_band'
    , 'loan_origination_principal_period'
    , 'borrower_language'
    , 'borrower_fico_bands_origination'
    , 'borrower_annual_income_band'
    , 'fdr_vs_aal_payment_variance_percentile'
    , 'fdr_vs_aal_payment_variance'
]


# Feature Selection (SelectKBest)
# Using f_classif for classification tasks
selector = SelectKBest(score_func=f_classif, k=20) # Select top 20 features
X_selected_kbest = selector.fit_transform(X, y)

# Get the names of selected features from SelectKBest
selected_kbest_feature_names = X.columns[selector.get_support(indices=True)]

# Combine SelectKBest features with the specified categorical features
# Ensure no duplicates and that all specified categorical features are included
all_selected_feature_names = list(selected_kbest_feature_names) + categorical_features_to_include
all_selected_feature_names = list(dict.fromkeys(all_selected_feature_names)) # Remove duplicates while preserving order


# Filter X to include only the combined set of selected features
X_selected = X[all_selected_feature_names]


# Model Selection and Training (RandomForestClassifier)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_selected, y)

# Identify Important Features
feature_importances = model.feature_importances_
feature_importances_series = pd.Series(feature_importances, index=X_selected.columns)
sorted_feature_importances = feature_importances_series.sort_values(ascending=False)

# Define columns to remove (including those previously specified by the user)
columns_to_remove = [
    'days_elapsed_origination_to_dq_30',
    'loan_investor',
    'loan_prepaid_flag',
    'loan_contributing_investor',
    'loan_no_payment_straight_to_chargeoff_flag',
    'loan_1st_pay_default_return_code',
    'days_elapsed_origination_to_dq',
    'principal_default_percent',
    'count_dpd_1_to_10',
    'count_dpd_cured',
    'count_dpd_30',
    'loan_chargeoff_flag',
    'loan_contractual_chargeoff_flag',
    'loan_in_debt_settlement_flag',
    'flag_dq',
    'flag_never_pay',
    'borrower_origination_state'
]


# Remove specified columns from feature importances
filtered_feature_importances = sorted_feature_importances.drop(columns_to_remove, errors='ignore')


print("Sorted Feature Importances (including specified categorical features and excluding others):")
display(filtered_feature_importances)

# Optional: Visualize filtered feature importances
plt.figure(figsize=(10, 8)) # Increased figure height for more features
sns.barplot(x=filtered_feature_importances.values, y=filtered_feature_importances.index, orient='h')
plt.title('Feature Importance for Predicting First Pay Default')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

# Visualize numerical feature default proportions
Visualize the contribution of each feature to the target variable `flag_1st_pay_default`.
## Subtask:
Create plots showing the proportion of defaulting customers within bins of the numerical features.

**Reasoning**:
To visualize the proportion of defaults within bins of numerical features, I will:
1. Identify numerical columns (excluding constant ones).
2. Iterate through each numerical column.
3. Create bins for the numerical feature.
4. Calculate the proportion of defaults within each bin.
5. Plot the default proportions against the bin centers or edges.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'df' and 'flag_1st_pay_default' are already loaded and available

# Identify numerical columns
numerical_cols = df.select_dtypes(include=np.number).columns
print(f"Initial numerical columns: {list(numerical_cols)}")

# Identify constant numerical columns using a for loop for robustness
constant_numerical_cols = []
for col in numerical_cols:
    try:
        # Exclude the target variable from the nunique check if it's numeric
        if col != 'flag_1st_pay_default' and df[col].nunique() == 1:
            constant_numerical_cols.append(col)
        # Handle the case where the target variable might be in numerical_cols
        elif col == 'flag_1st_pay_default':
             continue
    except Exception as e:
        print(f"Could not determine nunique for column '{col}': {e}")
print(f"Constant numerical columns: {constant_numerical_cols}")


# Define columns to exclude from plotting (keeping previous exclusions and ensuring no overlap)
columns_to_exclude_plotting = [
    'days_elapsed_origination_to_dq_30',
    'loan_investor', # Although listed, this is likely categorical. Will handle based on dtype.
    'loan_prepaid_flag',
    'loan_contributing_investor', # Although listed, this is likely categorical. Will handle based on dtype.
    'loan_no_payment_straight_to_chargeoff_flag',
    'loan_1st_pay_default_return_code', # Although listed, this is likely categorical. Will handle based on dtype.
    'days_elapsed_origination_to_dq',
    'principal_default_percent',
    'count_dpd_1_to_10',
    'count_dpd_cured',
    'count_dpd_30',
    'loan_chargeoff_flag',
    'loan_contractual_chargeoff_flag',
    'loan_in_debt_settlement_flag',
    'flag_dq',
    'flag_never_pay',
    'loan_bk_discharged_flag',
    'loan_deceased_flag'
]
print(f"Columns to exclude from plotting: {columns_to_exclude_plotting}")


# Filter out constant numerical columns, the target variable, and specified excluded columns
plottable_numerical_cols = [
    col for col in numerical_cols
    if col not in constant_numerical_cols and
       col != 'flag_1st_pay_default' and
       col not in columns_to_exclude_plotting
]
print(f"Plottable numerical columns after filtering: {plottable_numerical_cols}")


# Columns to impute with mean
columns_to_impute_mean = ['loan_application_NDI_ratio', 'loan_application_PTI_ratio']

print(f"Attempting to plot default proportions for {len(plottable_numerical_cols)} numerical columns.")

# Iterate through plottable numerical columns and create plots of default proportions
for col in plottable_numerical_cols:
    try:
        # Use the full df with the new cluster_label column
        temp_df = df[[col, 'flag_1st_pay_default']].copy() # Create a temporary df with only the relevant columns

        # Impute missing values for specified columns with their mean
        if col in columns_to_impute_mean:
            mean_value = temp_df[col].mean()
            if pd.isna(mean_value):
                print(f"Skipping plotting for column '{col}' because it contains only null values.")
                plt.close() # Ensure figure is closed
                continue
            temp_df[col].fillna(mean_value, inplace=True)
            print(f"Imputed missing values in '{col}' with mean ({mean_value:.2f}).")

        # Remove rows where the numerical column is NaN after imputation (only if still NaNs exist)
        temp_df.dropna(subset=[col, 'flag_1st_pay_default'], inplace=True)


        if temp_df.empty:
            print(f"Skipping plotting for column '{col}' due to no non-null data for plotting after imputation.")
            continue

        # Create bins for the numerical feature
        # Using qcut for quantiles to handle skewed data, fall back to cut if too many identical values
        try:
            # Attempt to create quantile bins and capture the bins information
            bins, intervals = pd.qcut(temp_df[col], q=10, labels=False, duplicates='drop', retbins=True)
            temp_df['bin'] = bins
            print(f"Created quantile bins for column '{col}'.")

        except ValueError as ve:
            print(f"Could not create quantile bins for '{col}' ({ve}), falling back to equal-width bins.")
            try:
                # Fall back to equal-width bins and capture the bins information
                bins, intervals = pd.cut(temp_df[col], bins=10, labels=False, include_lowest=True, retbins=True)
                temp_df['bin'] = bins
                if temp_df['bin'].isnull().all():
                     print(f"Skipping plotting for column '{col}' as equal-width binning resulted in all NaNs.")
                     continue
                print(f"Created equal-width bins for column '{col}'.")

            except Exception as cut_e:
                print(f"Skipping plotting for column '{col}' due to error during binning: {cut_e}")
                continue

        # Calculate the proportion of defaults and count of loans within each bin
        bin_analysis = temp_df.groupby('bin')['flag_1st_pay_default'].agg(['count', 'mean']).reset_index()
        bin_analysis = bin_analysis.rename(columns={'count': 'loan_count', 'mean': 'default_proportion'})

        # Calculate the percentage of total population for each bin
        total_loans_in_col = bin_analysis['loan_count'].sum()
        bin_analysis['percentage_of_total'] = (bin_analysis['loan_count'] / total_loans_in_col) * 100

        if bin_analysis.empty:
             print(f"Skipping plotting for column '{col}' as no bin analysis could be calculated after binning.")
             continue

        # Display bin information including edges, count, and percentage
        bin_info = pd.DataFrame({'Bin Index': bin_analysis['bin'],
                                 'Bin Edge': intervals[:-1] if len(intervals) == len(bin_analysis) + 1 else None, # Attempt to align edges
                                 'Loan Count': bin_analysis['loan_count'],
                                 'Percentage of Total': bin_analysis['percentage_of_total'],
                                 'Default Proportion': bin_analysis['default_proportion']
                                })
        print(f"Bin analysis for {col}:")
        display(bin_info)


        # Create a bar plot of default proportions
        plt.figure(figsize=(10, 6))
        sns.barplot(x=bin_analysis['bin'], y=bin_analysis['default_proportion'])
        plt.title(f'Proportion of First Pay Defaults by Bins of {col}')
        plt.xlabel(f'{col} Bin Index') # Using bin index as labels for now
        plt.ylabel('Proportion of Defaults')
        plt.show()
        plt.close()

    except Exception as e:
        print(f"Skipping plotting for column '{col}' due to error: {e}")
        plt.close()

In [ ]:
# Define the column to bin
column_to_bin = 'days_elapsed_negotiation'

# --- 2. Corrected Crosstab with Bin Limits ---
if f'{column_to_bin}_Decile_Label' in df.columns:

    print("\n## 📋 Crosstab of Decile Counts and Limits ##")

    # Calculate the count of loans in each descriptive decile bin
    decile_crosstab = df[f'{column_to_bin}_Decile_Label'].value_counts().sort_index().to_frame('Loan Count')

    # === FIX APPLIED HERE ===
    # Explicitly convert the CategoricalIndex back to an IntervalIndex
    # so we can access the .left and .right attributes.
    decile_crosstab.index = decile_crosstab.index.astype('interval')

    # Calculate the percentage of total loans
    total_loans = decile_crosstab['Loan Count'].sum()
    decile_crosstab['Percentage'] = (decile_crosstab['Loan Count'] / total_loans) * 100

    # Extract the lower and upper limit from the now-correct Interval Index
    decile_crosstab['Lower Limit'] = decile_crosstab.index.left
    decile_crosstab['Upper Limit'] = decile_crosstab.index.right

    # Reorder columns for clarity
    decile_crosstab = decile_crosstab[['Lower Limit', 'Upper Limit', 'Loan Count', 'Percentage']]

    display(decile_crosstab)

##Days Elapsed Negotiation Historgram

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'df' is your loaded DataFrame
column_to_bin = 'days_elapsed_negotiation'
decile_label_col = f'{column_to_bin}_Decile_Label'

# --- 1. Recreate Labeled Decile Bins and Crosstab (Necessary Setup) ---

temp_df = df.dropna(subset=[column_to_bin]).copy()

# A. Create the bins and intervals
try:
    decile_cut, intervals = pd.qcut(
        temp_df[column_to_bin],
        q=10,
        labels=False,
        duplicates='drop',
        retbins=True
    )
    # B. Create the descriptive labels
    decile_labels = pd.cut(
        temp_df[column_to_bin],
        bins=intervals,
        include_lowest=True,
        duplicates='drop',
        right=True
    )
    df[decile_label_col] = decile_labels

    # C. Create the Crosstab with Loan Counts
    decile_crosstab = df[decile_label_col].value_counts().sort_index().to_frame('Loan Count')

    # FIX: Convert the index back to IntervalIndex to extract limits
    decile_crosstab.index = decile_crosstab.index.astype('interval')

    # D. Prepare labels for the X-axis
    # Create descriptive labels like "[Lower - Upper]"
    decile_crosstab['Bin Label'] = (
        '[' + decile_crosstab.index.left.round(2).astype(str) +
        ' - ' +
        decile_crosstab.index.right.round(2).astype(str) +
        ']'
    )

    print(f"Decile Binning and Crosstab created for '{column_to_bin}'.")

except Exception as e:
    print(f"Skipping visualization due to error in binning: {e}")
    # Handle the error gracefully if binning fails
    exit() # Exit the block if setup fails


# --- 2. Bar Plot for Loan Count Frequency ---

print(f"\n## 📈 Bar Plot: Loan Count Frequency by Decile Bin ##")

plt.figure(figsize=(12, 6))

# Use the 'Bin Label' for the x-axis and 'Loan Count' for the y-axis
sns.barplot(
    x='Bin Label',
    y='Loan Count',
    data=decile_crosstab,
    palette="viridis"
)

plt.title(f'Loan Count Frequency by Decile Bins of {column_to_bin}')
plt.xlabel(f'{column_to_bin} Bin Range (Lower - Upper)')
plt.ylabel('Loan Count (Frequency)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'df' is your loaded DataFrame
column_to_bin = 'days_elapsed_negotiation'
segment_col = 'loan_vintage_month'

# --- 0. Prepare the segmentation column and unique values ---
if segment_col not in df.columns:
    print(f"Error: '{segment_col}' column not found in the DataFrame.")
    exit()

# Get and sort unique vintage months. Use np.sort() for robustness with datetime types.
vintage_months = df[segment_col].unique()
vintage_months = np.sort(vintage_months)

# --- Loop through each vintage month to create segmented visualizations ---
for vintage_month in vintage_months:
    print(f"\n=======================================================")
    print(f"## 🗓️ Processing Segment: {segment_col} = {vintage_month} ##")
    print(f"=======================================================")

    # 1. Filter the data for the current segment
    segment_df = df[df[segment_col] == vintage_month].copy()

    # Get the non-null data for the target column
    temp_data = segment_df.dropna(subset=[column_to_bin])[column_to_bin]

    # Check for minimum data points
    if temp_data.shape[0] < 10:
        print(f"Skipping visualization for {vintage_month}: Not enough non-null data points ({temp_data.shape[0]}) to create 10 deciles.")
        continue

    # --- 1. Decile Binning and Crosstab (PER SEGMENT) ---

    try:
        # A. Use pd.qcut to determine the bin edges (intervals)
        _, intervals = pd.qcut(
            temp_data,
            q=10,
            labels=False,
            duplicates='drop',
            retbins=True
        )

        # B. Use pd.cut with the determined intervals to create the IntervalIndex labels
        decile_labels = pd.cut(
            temp_data,
            bins=intervals,
            include_lowest=True,
            duplicates='drop',
            right=True
        )

        # C. Create the Crosstab with Loan Counts
        decile_crosstab = decile_labels.value_counts().sort_index().to_frame('Loan Count')

        # --- FIX: Explicitly convert the CategoricalIndex to IntervalIndex ---
        # This addresses the 'CategoricalIndex' object has no attribute 'left' error.
        if isinstance(decile_crosstab.index, pd.CategoricalIndex):
            decile_crosstab.index = decile_crosstab.index.astype('interval')


        # --- Check for Single Bin (Non-IntervalIndex) ---
        # Now this check should be for IntervalIndex, which covers all valid multi-bin cases.
        if not isinstance(decile_crosstab.index, pd.IntervalIndex):
            # Handle the true single-bin case (or remaining rare edge cases)
            if len(decile_crosstab) == 1:
                total_count = decile_crosstab['Loan Count'].iloc[0]
                unique_value = temp_data.iloc[0]

                decile_crosstab = pd.DataFrame({
                    'Bin Label': [f'All Values = {unique_value.round(2)}'],
                    'Loan Count': [total_count]
                })
                print(f"Plotting {vintage_month} as a single bin (value={unique_value.round(2)}).")
            else:
                print(f"Skipping visualization for {vintage_month}: Binning resulted in an unexpected Index type after conversion.")
                continue

        else:
            # D. Prepare labels for the X-axis (Standard Decile case)
            decile_crosstab['Bin Label'] = (
                '[' + decile_crosstab.index.left.round(2).astype(str) +
                ' - ' +
                decile_crosstab.index.right.round(2).astype(str) +
                ']'
            )
            print(f"Decile Binning and Crosstab created for '{column_to_bin}' in segment {vintage_month}.")


    except Exception as e:
        print(f"Skipping visualization due to error in binning for {vintage_month}: {e}")
        continue

    # --- 2. Bar Plot for Loan Count Frequency (PER SEGMENT) ---
    # This section now works for both deciles and the single-bin case.

    plt.figure(figsize=(12, 6))

    sns.barplot(
        x='Bin Label',
        y='Loan Count',
        data=decile_crosstab,
        palette="viridis"
    )

    plt.title(f'Loan Count Frequency by Decile Bins of {column_to_bin}\nSegment: {segment_col} = {vintage_month}')
    plt.xlabel('Bin Range or Value')
    plt.ylabel('Loan Count (Frequency)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# Assuming 'df' and the column names are defined as in the previous script
column_to_bin = 'days_elapsed_negotiation'
decile_label_col = f'{column_to_bin}_Decile_Label'

# Filter for non-null values for plotting
plot_df = df.dropna(subset=[column_to_bin, decile_label_col]).copy()

print(f"\n## 🎻 Violin Plot Visualization for {column_to_bin} ##")

plt.figure(figsize=(14, 7))
# Use the 'split=True' or 'inner="quartile"' parameter for more detail if needed.
sns.violinplot(x=decile_label_col, y=column_to_bin, data=plot_df, inner="quartile", palette="Set2")
plt.title(f'Violin Plot of {column_to_bin} by Decile (Full Distribution Shape)')
plt.xlabel('Decile Range (Lower, Upper]')
plt.ylabel(column_to_bin)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Assuming 'df' and the column names are defined as in the previous script
column_to_bin = 'days_elapsed_negotiation'
decile_label_col = f'{column_to_bin}_Decile_Label'

# Filter for non-null values for plotting
plot_df = df.dropna(subset=[column_to_bin, decile_label_col]).copy()

print(f"\n## 🎯 Strip Plot Visualization for {column_to_bin} ##")

plt.figure(figsize=(14, 7))
# 'jitter=True' spreads the points horizontally for visibility
sns.stripplot(x=decile_label_col, y=column_to_bin, data=plot_df, jitter=0.3, size=3, palette="Set1")
plt.title(f'Strip Plot (Raw Data Points) of {column_to_bin} by Decile')
plt.xlabel('Decile Range (Lower, Upper]')
plt.ylabel(column_to_bin)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
plt.close()

# Visualize categorical feature distributions by default status

## Subtask:
Create bar plots showing the count or proportion of defaulting customers within each category of the categorical features.


**Reasoning**:
Create bar plots showing the proportion of defaulting customers within each category of the categorical features to visualize their contribution to the target variable.



In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

# Define columns to exclude
columns_to_exclude = ['loan_id', 'loan_product', 'loan_investor', 'loan_contributing_investor']

# Filter out excluded columns
plottable_categorical_cols = [col for col in categorical_cols if col not in columns_to_exclude]


# Iterate through filtered categorical columns and create bar plots of default proportions
for col in plottable_categorical_cols:
    try:
        # Create a cross-tabulation with the target variable
        cross_tab = pd.crosstab(df[col], df['flag_1st_pay_default'])

        # Calculate the proportion of defaults within each category
        # Handle cases where a category might not have both default and non-default instances
        cross_tab['default_proportion'] = cross_tab.get(1, 0) / (cross_tab.get(0, 0) + cross_tab.get(1, 0))

        # Calculate loan count and percentage of total for each category
        cross_tab['loan_count'] = cross_tab.get(0, 0) + cross_tab.get(1, 0)
        total_loans = cross_tab['loan_count'].sum()
        cross_tab['percentage_of_total'] = (cross_tab['loan_count'] / total_loans) * 100

        # Sort the cross_tab index to order the x-axis
        cross_tab = cross_tab.sort_index()

        # Display table with categorical information
        display_table = cross_tab[['loan_count', 'percentage_of_total', 'default_proportion']].copy()
        print(f"Analysis for categorical feature: {col}")
        display(display_table)


        # Create a bar plot of default proportions
        plt.figure(figsize=(10, 6))
        sns.barplot(x=cross_tab.index, y=cross_tab['default_proportion'])
        plt.title(f'Proportion of First Pay Defaults by {col}')
        plt.xlabel(col)
        plt.ylabel('Proportion of Defaults')
        plt.xticks(rotation=90) # Rotate labels for readability
        plt.tight_layout()
        plt.show()
        plt.close() # Close the figure to manage memory
    except Exception as e:
        print(f"Skipping plotting for column '{col}' due to error: {e}")
        plt.close() # Ensure figure is closed even if plotting fails

# Visualize feature relationships

## Subtask:
Depending on the number of features and complexity, consider visualizing relationships between important features and the target variable using scatter plots or other relevant plot types.


**Reasoning**:
Identify important features, select pairs of features for visualization, and create scatter plots or box plots as appropriate, colored by the target variable, to visualize their relationship with the target variable.



In [ ]:
# Based on the previous feature importance analysis (filtered_feature_importances),
# let's select the top features for visualization.
# The top features identified were:
# days_elapsed_negotiation
# borrower_origination_risk_group
# loan_vintage_quarter
# loan_origination_ach_active_flag
# borrower_origination_risk_group_twentile

# Let's visualize the relationship between some pairs of these important features and the target.

# Pair 1: days_elapsed_negotiation (numerical) and borrower_origination_risk_group (numerical)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='days_elapsed_negotiation', y='borrower_origination_risk_group', hue='flag_1st_pay_default', alpha=0.5)
plt.title('Scatter Plot of Days Elapsed Negotiation vs. Borrower Risk Group by Default Status')
plt.xlabel('Days Elapsed Negotiation')
plt.ylabel('Borrower Origination Risk Group')
plt.show()

# Pair 2: days_elapsed_negotiation (numerical) and loan_origination_ach_active_flag (categorical/binary)
# Although loan_origination_ach_active_flag is binary numerical after encoding,
# it represents a categorical concept. A box plot is suitable here.
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='loan_origination_ach_active_flag', y='days_elapsed_negotiation', hue='flag_1st_pay_default')
plt.title('Box Plot of Days Elapsed Negotiation by ACH Active Flag and Default Status')
plt.xlabel('Loan Origination ACH Active Flag')
plt.ylabel('Days Elapsed Negotiation')
plt.xticks([0, 1], ['Inactive', 'Active']) # Label the x-axis
plt.show()

# Pair 3: loan_vintage_quarter (categorical) and borrower_origination_risk_group (numerical)
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for loan_vintage_quarter and sort them
vintage_quarter_order = sorted(df['loan_vintage_quarter'].unique())
sns.boxplot(data=df, x='loan_vintage_quarter', y='borrower_origination_risk_group', hue='flag_1st_pay_default', order=vintage_quarter_order)
plt.title('Box Plot of Borrower Risk Group by Loan Vintage Quarter and Default Status')
plt.xlabel('Loan Vintage Quarter')
plt.ylabel('Borrower Origination Risk Group')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 4: borrower_origination_risk_group_twentile (categorical) and days_elapsed_negotiation (numerical)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='borrower_origination_risk_group_twentile', y='days_elapsed_negotiation', hue='flag_1st_pay_default')
plt.title('Box Plot of Days Elapsed Negotiation by Borrower Risk Group Twentile and Default Status')
plt.xlabel('Borrower Origination Risk Group Twentile')
plt.ylabel('Days Elapsed Negotiation')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 5: loan_vintage_quarter (categorical) and applicant PTI
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for loan_vintage_quarter and sort them
vintage_quarter_order = sorted(df['loan_vintage_quarter'].unique())
sns.boxplot(data=df, x='loan_vintage_quarter', y='loan_application_PTI_ratio', hue='flag_1st_pay_default', order=vintage_quarter_order)
plt.title('Box Plot of Applicant PTI Ratio by Loan Vintage Quarter and Default Status')
plt.xlabel('Loan Vintage Quarter')
plt.ylabel('PTI Ratio')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 6: loan_vintage_quarter (categorical) and applicant NDI
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for loan_vintage_quarter and sort them
vintage_quarter_order = sorted(df['loan_vintage_quarter'].unique())
sns.boxplot(data=df, x='loan_vintage_quarter', y='loan_application_NDI_ratio', hue='flag_1st_pay_default', order=vintage_quarter_order)
plt.title('Box Plot of Applicant NDI Ratio by Loan Vintage Quarter and Default Status')
plt.xlabel('Loan Vintage Quarter')
plt.ylabel('NDI Ratio')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 7: loan_vintage_quarter (categorical) and days elapsed negotiation
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for loan_vintage_quarter and sort them
vintage_quarter_order = sorted(df['loan_vintage_quarter'].unique())
sns.boxplot(data=df, x='loan_vintage_quarter', y='days_elapsed_negotiation', hue='flag_1st_pay_default', order=vintage_quarter_order)
plt.title('Box Plot of Days Elapsed Negotiation by Loan Vintage Quarter and Default Status')
plt.xlabel('Loan Vintage Quarter')
plt.ylabel('Days Elapsed Negotiation')
# Set the y-axis limit from 0 to 400
plt.ylim(0, 400)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 7: origination risk group and days elapsed negotiation
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for risk group and sort them
risk_group_order = sorted(df['borrower_origination_risk_group'].unique())
sns.boxplot(data=df, x='borrower_origination_risk_group', y='days_elapsed_negotiation', hue='flag_1st_pay_default', order=risk_group_order)
plt.title('Box Plot of Days Elapsed Negotiation by Risk Group and Default Status')
plt.xlabel('Borrower Risk Group')
plt.ylabel('Days Elapsed Negotiation')
# Set the y-axis limit from 0 to 400
plt.ylim(0, 400)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 8: origination risk group and ADR vs AAL Pay Variance
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for risk group and sort them
risk_group_order = sorted(df['borrower_origination_risk_group'].unique())
sns.boxplot(data=df, x='borrower_origination_risk_group', y='fdr_vs_aal_payment_variance', hue='flag_1st_pay_default', order=risk_group_order)
plt.title('Box Plot of Payment Variance Dollars by Risk Group and Default Status')
plt.xlabel('Borrower Risk Group')
plt.ylabel('Pay Variance Dollars')
# Set the y-axis limit from -300 to 500
plt.ylim(-300, 500)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 9: loan_vintage_quarter (categorical) and ADR vs AAL Pay Variance
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
# Get the unique values for loan_vintage_quarter and sort them
vintage_quarter_order = sorted(df['loan_vintage_quarter'].unique())
sns.boxplot(data=df, x='loan_vintage_quarter', y='fdr_vs_aal_payment_variance', hue='flag_1st_pay_default', order=vintage_quarter_order)
plt.title('Box Plot of Pay Variance Dollars by Loan Vintage Quarter and Default Status')
plt.xlabel('Loan Vintage Quarter')
plt.ylabel('Pay Variance Dollars')
# Set the y-axis limit from -300 to 500
plt.ylim(-300, 500)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pair 10: origination risk group and ADR vs AAL Pay Variance
# Use a box plot or violin plot to show the distribution of risk group across vintage quarters,
# separated by default status.
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='fdr_vs_aal_payment_variance', y='days_elapsed_negotiation', hue='flag_1st_pay_default', alpha=0.5)
plt.title('Scatter Plot of Days Elapsed in Negotiation by ADR vs AAL Pay Variance by Default Status')
plt.xlabel('ADR vs AAL Pay Variance')
plt.ylabel('Days Elapsed in Negotiation')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Classification/Clustering
Analyze the provided data to identify and describe the types of customers who are most likely to default on their first loan payment, using the `flag_1st_pay_default` column as the target variable. This analysis should involve either classification or clustering techniques to understand the characteristics of these customers.

## Data preprocessing

### Subtask:
Ensure the data is ready for classification or clustering by handling missing values, encoding categorical variables, and scaling numerical features. We can reuse the preprocessing steps from the previous task.


**Reasoning**:
The data has already been loaded and partially preprocessed in the previous steps. I will now continue with the remaining preprocessing steps as outlined in the instructions to prepare the data for either classification or clustering. This involves separating the target variable, handling missing values, encoding categorical features, dropping date/dbdate columns, and checking for constant columns and remaining NaNs.



In [ ]:
# Separate target variable
X = df.drop('flag_1st_pay_default', axis=1)
y = df['flag_1st_pay_default']

# Drop columns with all missing values
X.dropna(axis=1, how='all', inplace=True)

# Identify categorical and numerical columns after dropping
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=np.number).columns

# Impute missing values (using mean for numerical and most frequent for categorical)
numerical_imputer = SimpleImputer(strategy='mean')
X[numerical_cols] = numerical_imputer.fit_transform(X[numerical_cols])

categorical_imputer = SimpleImputer(strategy='most_frequent')
X[categorical_cols] = categorical_imputer.fit_transform(X[categorical_cols])

# Encode categorical features
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Drop date and dbdate columns before feature selection
date_cols = X.select_dtypes(include=['datetime64[ns]', 'dbdate']).columns
X = X.drop(columns=date_cols)

# Check for constant columns before feature selection
constant_cols = [col for col in X.columns if X[col].nunique() == 1]
print(f"Constant columns remaining: {constant_cols}")

# Check for remaining NaNs before feature selection
print("Number of NaNs per column after imputation:")
display(X.isnull().sum())

### 🛡️ Clean Split-Workflow: Separating Pre-Origination Features from Post-Origination Leakage
To prevent target leakage during predictive tasks (like model development or unsupervised clustering), we partition our feature space `X` immediately after imputation:
* **`X_pre_origination`**: Only contains variables known *prior* to loan origination. This is what we will feed into our clustering and classification models.
* **`X_post_origination`**: Reassigned to a secondary dataframe reserved strictly for diagnostic lookup, profiling, and post-hoc evaluation.

In [ ]:
# 1. Define COMPLETE master list of post-origination / leakage columns
post_origination_cols = [
    'days_elapsed_origination_to_dq',
    'days_elapsed_origination_to_dq_30',
    'count_dpd_30',
    'count_dpd_1_to_10',
    'flag_dq',
    'loan_prepaid_flag',
    'loan_1st_pay_default_return_code',
    'count_dpd_cured',
    'loan_no_payment_straight_to_chargeoff_flag',
    'loan_chargeoff_flag',
    'loan_contractual_chargeoff_flag',
    'flag_never_pay',
    'principal_default_percent',
    'loan_only_one_transaction_day_of_week',
    'loan_in_debt_settlement_flag'
]

# 2. Extract constant columns (excluding these prevents zero-variance scaling issues)
constant_cols = ['loan_product', 'loan_application_loan_use', 'loan_fraud_flag', 'loan_bk_discharged_flag']

# 3. Create clean predictive (pre-origination) dataset
pre_orig_cols = [col for col in X.columns if col not in ['loan_id'] + constant_cols + post_origination_cols]
X_pre_origination = X[pre_orig_cols]

# 4. Relegate diagnostic metadata (post-origination) safely to its own matrix
available_post_orig = [col for col in post_origination_cols if col in X.columns]
X_post_origination = X[available_post_orig]

print(f"[SUCCESS] Clean Split-Workflow complete.")
print(f"  Ⅳ Clean pre-origination variables for modeling: {X_pre_origination.shape[1]} columns")
print(f"  Ⅳ Post-origination variables retained for profiling: {X_post_origination.shape[1]} columns")

## Clustering analysis (for identifying groups of customers)

### Subtask:
Apply clustering algorithms to the customer data (excluding the target variable) to identify different customer segments.


**Reasoning**:
Apply clustering algorithms to the customer data to identify different customer segments.



In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Pre-origination only (no post-origination leakage)
X_clustering = X_pre_origination.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

X['cluster_label'] = cluster_labels
df['cluster_label'] = cluster_labels

# Refresh rates for downstream summary / PCA naming (do not use stale leaky rates from earlier cells)
cluster_default_rates = df.groupby('cluster_label')['flag_1st_pay_default'].mean()

display(df['cluster_label'].value_counts())
display(cluster_default_rates)


## Analyze clusters

### Subtask:
Examine the characteristics of the customers within each cluster and see if any clusters have a significantly higher proportion of `flag_1st_pay_default`.


**Reasoning**:
Group the original dataframe by the cluster label, calculate the mean default rate for each cluster, and examine the mean or mode of relevant features to characterize each cluster.



In [ ]:
# Sync cluster labels from modeling matrix (pre-origination K-Means above)
df['cluster_label'] = X['cluster_label']

cluster_default_rates = df.groupby('cluster_label')['flag_1st_pay_default'].mean()

print("Default Rate by Cluster:")
display(cluster_default_rates)

relevant_features = ['loan_vintage_quarter', 'borrower_origination_risk_group',
                     'days_elapsed_negotiation', 'loan_origination_loan_amount',
                     'loan_origination_interest_rate', 'loan_origination_term_in_months',
                     'borrower_fico_bands_origination', 'borrower_annual_income_band',
                     'avg_monthly_draft', 'monthly_loan_payment_amount',
                     'loan_origination_loan_amount_band', 'loan_origination_interest_rate_band',
                     'borrower_fico_band_category_origination', 'borrower_origination_risk_group_twentile',
                     'borrower_home_owner_flag', 'borrower_origination_state', 'co_borrower_flag',
                     'loan_origination_first_payment_day_of_week', 'loan_only_one_transaction_day_of_week',
                     'days_elapsed_bucket_negotiation', 'fdr_vs_aal_payment_variance',
                     'fdr_vs_aal_payment_variance_percent_change',
                     'fdr_vs_aal_payment_variance_percentile', 'loan_application_NDI_ratio',
                     'loan_application_PTI_ratio']

numerical_features = df[relevant_features].select_dtypes(include='number').columns
categorical_features = df[relevant_features].select_dtypes(include=['object', 'category']).columns

numerical_characteristics_mean = df.groupby('cluster_label')[numerical_features].mean()
categorical_characteristics_mode = df.groupby('cluster_label')[categorical_features].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)

print("\nCluster Characteristics (Mean for Numerical Features):")
display(numerical_characteristics_mean)

print("\nCluster Characteristics (Mode for Categorical Features):")
display(categorical_characteristics_mode)

high_default_clusters = cluster_default_rates[cluster_default_rates > cluster_default_rates.mean()].index
print("\nCustomer Profiles in High-Default Clusters:")
for cluster_id in high_default_clusters:
    print(f"\nCluster {cluster_id} (Default Rate: {cluster_default_rates.loc[cluster_id]:.4f}):")
    display(numerical_characteristics_mean.loc[cluster_id])


In [ ]:
# --- Cluster alignment: one population for PCA + summary (run after K-Means + analyze cell above) ---
assert len(df) == len(X_pre_origination), "df and X_pre_origination row counts must match"

cluster_default_rates = df.groupby('cluster_label')['flag_1st_pay_default'].mean()
cluster_counts = df['cluster_label'].value_counts().sort_index()
print(f"Population: {len(df):,} loans | clusters: {dict(cluster_counts)}")
print("FPD rate by cluster (same labels used in PCA + summary):")
display((cluster_default_rates * 100).round(3))

# Requires numerical_characteristics_mean from the cell above
if 'numerical_characteristics_mean' not in globals():
    raise RuntimeError("Run the 'Analyze clusters' cell above first (builds numerical_characteristics_mean).")

clusters = sorted(df['cluster_label'].unique())
cluster_assignment = {}

critical_id = cluster_default_rates.idxmax()
cluster_assignment[critical_id] = {
    "name": "CRITICAL - Ultra-High Default Segment",
    "description": "Highest FPD rate in this vintage (pre-origination clustering).",
}

remaining = [c for c in clusters if c != critical_id]
premium_id = numerical_characteristics_mean.loc[remaining, 'loan_origination_loan_amount'].idxmax()
cluster_assignment[premium_id] = {
    "name": "Low Risk, High-Value Premium Loans",
    "description": "Largest average loan amounts among remaining clusters.",
}
remaining = [c for c in remaining if c != premium_id]
higher_risk_id = numerical_characteristics_mean.loc[remaining, 'borrower_origination_risk_group'].idxmax()
cluster_assignment[higher_risk_id] = {
    "name": "Higher Risk, Mid-Sized Loans (Least Homeowners)",
    "description": "Higher risk group among remaining clusters.",
}
remaining = [c for c in remaining if c != higher_risk_id]
if remaining:
    cluster_assignment[remaining[0]] = {
        "name": "Moderate-Low Risk, Mid-Sized Loans",
        "description": "Remaining pre-origination segment.",
    }

print("cluster_label → profile name (for plots + summary):")
for cid in clusters:
    print(f"  {cid}: {cluster_assignment[cid]['name']} (FPD {cluster_default_rates.loc[cid]*100:.2f}%)")


## Summarize customer types

### Subtask:
Based on the clustering analysis, describe the typical characteristics of customers who tend to default on their first payment.


**Reasoning**:
Synthesize the findings from the cluster analysis to describe the typical characteristics of customers in the high-default cluster (Cluster 1).



In [ ]:
print("Typical characteristics of customers in the high-default cluster (Cluster 0):")
print("- Default Rate: {:.4f}".format(cluster_default_rates.loc[0]))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'borrower_origination_risk_group']))
print("- Borrower NDI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'loan_application_NDI_ratio']))
print("- Borrower PTI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'loan_application_PTI_ratio']))
print("- Days Elapsed Negotiation (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'days_elapsed_negotiation']))
print("- Loan Origination Loan Amount (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'loan_origination_loan_amount']))
print("- Loan Origination Interest Rate (Mean): {:.4f}".format(numerical_characteristics_mean.loc[0, 'loan_origination_interest_rate']))
print("- Borrower Home Owner Flag (Mean): {:.2f} (Lower value indicates less likely to be homeowner)".format(numerical_characteristics_mean.loc[0, 'borrower_home_owner_flag']))
print("- Borrower Origination Risk Group Twentile (Mode): {}".format(categorical_characteristics_mode.loc[0, 'borrower_origination_risk_group_twentile']))
print("- ADR vs AAL Pay Variance (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'fdr_vs_aal_payment_variance']))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[0, 'borrower_origination_risk_group']))
print("- FICO Band Origination (Mode): {}".format(categorical_characteristics_mode.loc[0, 'borrower_fico_bands_origination']))
print("- Vintage Quarter (Mode): {}".format(categorical_characteristics_mode.loc[0, 'loan_vintage_quarter']))

In [ ]:
print("Typical characteristics of customers in the high-default cluster (Cluster 1):")
print("- Default Rate: {:.4f}".format(cluster_default_rates.loc[1]))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'borrower_origination_risk_group']))
print("- Borrower NDI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'loan_application_NDI_ratio']))
print("- Borrower PTI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'loan_application_PTI_ratio']))
print("- Days Elapsed Negotiation (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'days_elapsed_negotiation']))
print("- Loan Origination Loan Amount (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'loan_origination_loan_amount']))
print("- Loan Origination Interest Rate (Mean): {:.4f}".format(numerical_characteristics_mean.loc[1, 'loan_origination_interest_rate']))
print("- Borrower Home Owner Flag (Mean): {:.2f} (Lower value indicates less likely to be homeowner)".format(numerical_characteristics_mean.loc[1, 'borrower_home_owner_flag']))
print("- Borrower Origination Risk Group Twentile (Mode): {}".format(categorical_characteristics_mode.loc[1, 'borrower_origination_risk_group_twentile']))
print("- ADR vs AAL Pay Variance (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'fdr_vs_aal_payment_variance']))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[1, 'borrower_origination_risk_group']))
print("- FICO Band Origination (Mode): {}".format(categorical_characteristics_mode.loc[1, 'borrower_fico_bands_origination']))
print("- Vintage Quarter (Mode): {}".format(categorical_characteristics_mode.loc[1, 'loan_vintage_quarter']))

In [ ]:
print("Typical characteristics of customers in the high-default cluster (Cluster 2):")
print("- Default Rate: {:.4f}".format(cluster_default_rates.loc[2]))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'borrower_origination_risk_group']))
print("- Borrower NDI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'loan_application_NDI_ratio']))
print("- Borrower PTI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'loan_application_PTI_ratio']))
print("- Days Elapsed Negotiation (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'days_elapsed_negotiation']))
print("- Loan Origination Loan Amount (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'loan_origination_loan_amount']))
print("- Loan Origination Interest Rate (Mean): {:.4f}".format(numerical_characteristics_mean.loc[2, 'loan_origination_interest_rate']))
print("- Borrower Home Owner Flag (Mean): {:.2f} (Lower value indicates less likely to be homeowner)".format(numerical_characteristics_mean.loc[2, 'borrower_home_owner_flag']))
print("- Borrower Origination Risk Group Twentile (Mode): {}".format(categorical_characteristics_mode.loc[2, 'borrower_origination_risk_group_twentile']))
print("- ADR vs AAL Pay Variance (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'fdr_vs_aal_payment_variance']))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[2, 'borrower_origination_risk_group']))
print("- FICO Band Origination (Mode): {}".format(categorical_characteristics_mode.loc[2, 'borrower_fico_bands_origination']))
print("- Vintage Quarter (Mode): {}".format(categorical_characteristics_mode.loc[2, 'loan_vintage_quarter']))

In [ ]:
print("Typical characteristics of customers in the high-default cluster (Cluster 3):")
print("- Default Rate: {:.4f}".format(cluster_default_rates.loc[3]))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'borrower_origination_risk_group']))
print("- Borrower NDI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'loan_application_NDI_ratio']))
print("- Borrower PTI (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'loan_application_PTI_ratio']))
print("- Days Elapsed Negotiation (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'days_elapsed_negotiation']))
print("- Loan Origination Loan Amount (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'loan_origination_loan_amount']))
print("- Loan Origination Interest Rate (Mean): {:.4f}".format(numerical_characteristics_mean.loc[3, 'loan_origination_interest_rate']))
print("- Borrower Home Owner Flag (Mean): {:.2f} (Lower value indicates less likely to be homeowner)".format(numerical_characteristics_mean.loc[3, 'borrower_home_owner_flag']))
print("- Borrower Origination Risk Group Twentile (Mode): {}".format(categorical_characteristics_mode.loc[3, 'borrower_origination_risk_group_twentile']))
print("- ADR vs AAL Pay Variance (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'fdr_vs_aal_payment_variance']))
print("- Borrower Risk Group (Mean): {:.2f}".format(numerical_characteristics_mean.loc[3, 'borrower_origination_risk_group']))
print("- FICO Band Origination (Mode): {}".format(categorical_characteristics_mode.loc[3, 'borrower_fico_bands_origination']))
print("- Vintage Quarter (Mode): {}".format(categorical_characteristics_mode.loc[3, 'loan_vintage_quarter']))

## Summary:

### Data Analysis Key Findings
* The clustering analysis identified four distinct customer clusters.
* The highest first-pay default rate (approximately 7.4%) was Cluster 2 which was significantly higher compared to the other clusters (below 1%).
* Above average ADR vs AAL payment variance ($\sim$82.8)
* Higher average borrower origination risk group ($\sim$11.95)
* Above average borrower NDI
* Above average borrower PTI
* Longer average negotiation periods ($\sim$61.42 days)
* Smaller average loan amounts ($\sim$\$14909)
* Slightly below average interest rates ($\sim$23.60%).
* Customers are less likely to be homeowners (mean `borrower_home_owner_flag` $\sim$0.43)
* The mode of the vintage quarter from this cluster is 2023 Q2


### Insights or Next Steps
* Focus on customers with characteristics similar to Cluster 2 for targeted risk assessment and mitigation strategies.


## Summarize visualizations

### Subtask:
Briefly summarize the key insights gained from the visualizations regarding the features that appear to have the most significant impact on `flag_1st_pay_default`.


**Reasoning**:
Summarize the key insights from the visualizations to identify the features with the most significant impact on `flag_1st_pay_default`.



In [ ]:
print("Summary of key insights from visualizations:")
print("\nNumerical Features:")
print("- 'days_elapsed_negotiation': Histograms and scatter plots suggest that a higher number of days elapsed in negotiation is associated with a higher likelihood of first pay default.")
print("- 'borrower_origination_risk_group': Both histograms and scatter/box plots indicate that higher borrower risk groups have a higher proportion of defaults.")
print("- Other numerical features like loan amount, interest rate, and payment amounts show some distributional differences between default and non-default groups, but the separation is less distinct compared to 'days_elapsed_negotiation' and 'borrower_origination_risk_group'.")

print("\nCategorical Features:")
print("- 'loan_vintage_quarter': While not consistently showing a strong pattern across all quarters, some quarters might have slightly higher default proportions. The box plots show some variation in risk group distribution across quarters for defaulting loans.")
print("- 'borrower_origination_ach_active_flag': The box plot suggests a difference in 'days_elapsed_negotiation' distribution based on this flag and default status, indicating its relevance.")
print("- 'borrower_origination_risk_group_twentile': Similar to the main risk group, certain risk twentiles appear to have higher default rates, as suggested by the box plots with 'days_elapsed_negotiation'.")
print("- 'loan_1st_pay_default_return_code': The bar plot (if successfully generated) would show which return codes are most frequent among defaulting loans, likely highlighting reasons like 'Insufficient Funds'. (Note: Plotting for this feature had issues, but the variable description suggests its importance).")

print("\nFeature Relationships:")
print("- The scatter plot of 'days_elapsed_negotiation' vs. 'borrower_origination_risk_group' shows a tendency for defaulting loans to cluster in regions with higher values for both features, although there is significant overlap.")
print("- Box plots highlight how the distribution of numerical features like 'days_elapsed_negotiation' and 'borrower_origination_risk_group' varies across categories of other features and by default status, reinforcing the importance of these interactions.")

## Summary:

### Data Analysis Key Findings

*   **Numerical Features:**
    *   `days_elapsed_negotiation` and `borrower_origination_risk_group` showed the strongest association with default, with higher values in both features correlating with a higher likelihood of first pay default.
    *   Other numerical features exhibited less distinct differences between defaulting and non-defaulting groups.
    *   Some numerical features (`loan_application_NDI_ratio` and `loan_application_PTI_ratio`) could not be plotted with the target variable due to data distribution issues.
*   **Categorical Features:**
    *   `borrower_origination_risk_group_twentile` and `borrower_origination_ach_active_flag` appear relevant, with certain categories showing differing default proportions or distributions of related numerical features for defaulting loans.
    *   `loan_vintage_quarter` showed some variation in default rates across quarters, and the distribution of risk groups within defaulting loans varied by quarter.
    *   Several categorical features (`loan_application_lead_category`, `loan_application_lead_restrictions`, `loan_application_customer_experience`, `loan_application_utm_lead_source`, `loan_application_utm_lead_channel`, `loan_application_loan_use`, and `loan_1st_pay_default_return_code`) could not be plotted due to data issues where one of the target classes was missing within some categories.
*   **Feature Relationships:**
    *   Defaulting loans tended to cluster in regions of the scatter plot with higher values for both `days_elapsed_negotiation` and `borrower_origination_risk_group`.
    *   Box plots revealed important interactions, showing how distributions of key numerical features varied across categories of other features based on default status.

### Insights or Next Steps

*   Further investigation into the data issues preventing plotting for certain categorical and numerical features is needed to understand their potential contribution to the target variable.
*   The identified strong predictors (`days_elapsed_negotiation` and `borrower_origination_risk_group`) and their interactions should be prioritized in subsequent modeling steps.


# Visualize clusters using important features
Visualize the clustering results to understand the characteristics of the identified customer segments and how they relate to first pay defaults.

### Subtask:
Create scatter plots or other relevant plot types using combinations of the previously identified important features, coloring the data points by their assigned cluster label.


**Reasoning**:
Identify the top features and create scatter plots and box plots to visualize the clusters using combinations of these features.



In [ ]:
# Identify the top features based on the previous analysis.
# Let's use the top features from the filtered_feature_importances series.
top_features = filtered_feature_importances.index.tolist()

# Select a few pairs of important features for visualization.
# We'll choose a mix of numerical and categorical features.
# Ensure the features selected are present in the dataframe used for clustering (df).

# Check if the top features list is empty or contains sufficient features for pairing.
if len(top_features) < 2:
    print("Not enough important features identified for pairing and visualization.")
else:
    # Pair 1: Numerical vs Numerical
    if 'days_elapsed_negotiation' in top_features and 'borrower_origination_risk_group' in top_features:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df, x='days_elapsed_negotiation', y='borrower_origination_risk_group', hue='cluster_label', alpha=0.5, palette='viridis')
        plt.title('Cluster Visualization: Days Elapsed Negotiation vs. Borrower Risk Group')
        plt.xlabel('Days Elapsed Negotiation')
        plt.ylabel('Borrower Origination Risk Group')
        plt.show()
        plt.close()

    # Pair 2: Numerical vs Categorical
    if 'days_elapsed_negotiation' in top_features and 'borrower_origination_risk_group_twentile' in top_features:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df, x='borrower_origination_risk_group_twentile', y='days_elapsed_negotiation', hue='cluster_label', palette='viridis')
        plt.title('Cluster Visualization: Days Elapsed Negotiation by Borrower Risk Group Twentile')
        plt.xlabel('Borrower Origination Risk Group Twentile')
        plt.ylabel('Days Elapsed Negotiation')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        plt.close()

    # Pair 3: Categorical vs Categorical (Proportion of clusters within a categorical feature)
    if 'loan_vintage_quarter' in top_features:
        # Calculate the proportion of each cluster within each vintage quarter
        cluster_vintage_crosstab = pd.crosstab(df['loan_vintage_quarter'], df['cluster_label'], normalize='index')

        cluster_vintage_crosstab.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
        plt.title('Cluster Distribution by Loan Vintage Quarter')
        plt.xlabel('Loan Vintage Quarter')
        plt.ylabel('Proportion within Quarter')
        plt.xticks(rotation=45)
        plt.legend(title='Cluster Label', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        plt.close()

    # Pair 4: Numerical vs Numerical (Another pair from top features)
    # Check if there's at least a third feature to pair
    if len(top_features) >= 3 and top_features[0] != top_features[2] and top_features[1] != top_features[2]:
        feature1 = top_features[0]
        feature2 = top_features[2]
        # Ensure both selected features are numerical for a scatter plot
        if pd.api.types.is_numeric_dtype(df[feature1]) and pd.api.types.is_numeric_dtype(df[feature2]):
            plt.figure(figsize=(10, 6))
            sns.scatterplot(data=df, x=feature1, y=feature2, hue='cluster_label', alpha=0.5, palette='viridis')
            plt.title(f'Cluster Visualization: {feature1} vs. {feature2}')
            plt.xlabel(feature1)
            plt.ylabel(feature2)
            plt.show()
            plt.close()
        else:
            print(f"Skipping scatter plot for {feature1} and {feature2} as they are not both numerical.")


**Reasoning**:
The error indicates that the 'cluster_label' column is not present in the `df` DataFrame when the plotting code is executed. This is likely because the clustering code cell, which adds this column, was not run before the plotting cell in the current session. I need to re-run the clustering code cell to add the 'cluster_label' column to the `df` DataFrame.



In [ ]:
# Profiling frame (same cluster labels as pre-origination K-Means above)
df_profile = df.copy()
df_profile['cluster_label_clean'] = df_profile['cluster_label']
print(f"Profiling population: {len(df_profile):,} loans (full vintage cohort)")


**Reasoning**:
The 'cluster_label' column has now been added to the `df` DataFrame. I can proceed with creating the cluster visualization plots using the top features as specified in the subtask.



In [ ]:
# Identify the top features based on the previous analysis.
# Let's use the top features from the filtered_feature_importances series.
top_features = filtered_feature_importances.index.tolist()

# Select a few pairs of important features for visualization.
# We'll choose a mix of numerical and categorical features.
# Ensure the features selected are present in the dataframe used for clustering (df).

# Check if the top features list is empty or contains sufficient features for pairing.
if len(top_features) < 2:
    print("Not enough important features identified for pairing and visualization.")
else:
    # Pair 1: Numerical vs Numerical
    if 'days_elapsed_negotiation' in top_features and 'borrower_origination_risk_group' in top_features:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=df, x='days_elapsed_negotiation', y='borrower_origination_risk_group', hue='cluster_label', alpha=0.5, palette='viridis')
        plt.title('Cluster Visualization: Days Elapsed Negotiation vs. Borrower Risk Group')
        plt.xlabel('Days Elapsed Negotiation')
        plt.ylabel('Borrower Origination Risk Group')
        plt.show()
        plt.close()

    # Pair 2: Numerical vs Categorical
    if 'days_elapsed_negotiation' in top_features and 'borrower_origination_risk_group_twentile' in top_features:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df, x='borrower_origination_risk_group_twentile', y='days_elapsed_negotiation', hue='cluster_label', palette='viridis')
        plt.title('Cluster Visualization: Days Elapsed Negotiation by Borrower Risk Group Twentile')
        plt.xlabel('Borrower Origination Risk Group Twentile')
        plt.ylabel('Days Elapsed Negotiation')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        plt.close()

    # Pair 3: Categorical vs Categorical (Proportion of clusters within a categorical feature)
    if 'loan_vintage_quarter' in top_features:
        # Calculate the proportion of each cluster within each vintage quarter
        cluster_vintage_crosstab = pd.crosstab(df['loan_vintage_quarter'], df['cluster_label'], normalize='index')

        cluster_vintage_crosstab.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
        plt.title('Cluster Distribution by Loan Vintage Quarter')
        plt.xlabel('Loan Vintage Quarter')
        plt.ylabel('Proportion within Quarter')
        plt.xticks(rotation=45)
        plt.legend(title='Cluster Label', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        plt.close()

    # Pair 4: Numerical vs Numerical (Another pair from top features)
    # Check if there's at least a third feature to pair
    if len(top_features) >= 3 and top_features[0] != top_features[2] and top_features[1] != top_features[2]:
        feature1 = top_features[0]
        feature2 = top_features[2]
        # Ensure both selected features are numerical for a scatter plot
        if pd.api.types.is_numeric_dtype(df[feature1]) and pd.api.types.is_numeric_dtype(df[feature2]):
            plt.figure(figsize=(10, 6))
            sns.scatterplot(data=df, x=feature1, y=feature2, hue='cluster_label', alpha=0.5, palette='viridis')
            plt.title(f'Cluster Visualization: {feature1} vs. {feature2}')
            plt.xlabel(feature1)
            plt.ylabel(feature2)
            plt.show()
            plt.close()
        else:
            print(f"Skipping scatter plot for {feature1} and {feature2} as they are not both numerical.")

### 🛡️ Clean Split & Workflow Verification
Let's run a quick diagnostic check to verify that all datasets are loaded in memory, aligned, and completely free of target leakage.

In [ ]:
# Verify memory state and target leakage separation
print("=== Workspace Diagnostic ===")
print(f"✓ Active Base DataFrame (df): {df.shape[0]} rows, {df.shape[1]} columns")
print(f"✓ Pre-origination clean features (X_pre_origination): {X_pre_origination.shape[1]} columns")
print(f"✓ Post-origination diagnostic lookup (X_post_origination): {X_post_origination.shape[1]} columns")

# Built here so "Run all" works before later SHAP / logit cells
if 'X_selected' not in globals():
    raise NameError("X_selected is missing — run the feature-selection cells above first.")
X_selected_clean = X_selected.drop(columns=post_origination_cols, errors='ignore')
print(f"✓ Selected modeling features (X_selected_clean): {X_selected_clean.shape[1]} columns")

# Quick leakage assertion check
leakage_detected = any(col in X_pre_origination.columns for col in post_origination_cols)
print(f"🛡️ Target Leakage in Training Features? {'🚨 YES (Warning!)' if leakage_detected else '✅ NO (Clean Split)'}")


## Visualize clusters using dimensionality reduction

### Subtask:
Apply a dimensionality reduction technique (like PCA) and then create scatter plots of the reduced dimensions, colored by cluster label.


**Reasoning**:
Apply a dimensionality reduction technique (like PCA) and then create scatter plots of the reduced dimensions, colored by cluster label.



In [ ]:
from sklearn.decomposition import PCA

# Uses X_scaled from pre-origination K-Means and cluster_assignment from alignment cell above
if 'cluster_assignment' not in globals():
    raise RuntimeError("Run the Cluster alignment cell before PCA (builds cluster_assignment).")

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

X_pca_df = pd.DataFrame(X_pca, columns=['principal_component_1', 'principal_component_2'])
X_pca_df['cluster_label'] = df['cluster_label'].values
label_mapping = {k: v['name'] for k, v in cluster_assignment.items()}
X_pca_df['cluster_profile'] = X_pca_df['cluster_label'].map(label_mapping)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='principal_component_1',
    y='principal_component_2',
    hue='cluster_profile',
    data=X_pca_df,
    palette='Dark2',
    alpha=0.5,
)
plt.title('Principal Component Analysis of Customer Data, Colored by Risk Profile')
plt.xlabel('PC1: Loan Scale & Monthly Payment Size')
plt.ylabel('PC2: Risk Tier & Pricing')
plt.legend(title='Cluster Profile', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Get the correct feature names used for the clean PCA scaling
feature_names = X_pre_origination.columns.tolist()

# Extract loadings (components_) from the fitted PCA model
# Note: pca was already fit in cell 4e038a2f
loadings = pca.components_

# Create a DataFrame to easily view the feature contributions
loadings_df = pd.DataFrame(
    loadings.T,
    columns=['PC1_Loading', 'PC2_Loading'],
    index=feature_names
)

# Display top features contributing to PC1
print("=== Top 10 Positive Contributors to PC1 ===")
display(loadings_df['PC1_Loading'].sort_values(ascending=False).head(10))

print("\n=== Top 10 Negative Contributors to PC1 ===")
display(loadings_df['PC1_Loading'].sort_values(ascending=True).head(10))

# Display top features contributing to PC2
print("\n=== Top 10 Positive Contributors to PC2 ===")
display(loadings_df['PC2_Loading'].sort_values(ascending=False).head(10))

print("\n=== Top 10 Negative Contributors to PC2 ===")
display(loadings_df['PC2_Loading'].sort_values(ascending=True).head(10))

In [ ]:
# Let's print the top 10 positive and negative contributors of both components side-by-side to understand PC2.
pc_summary = pd.DataFrame({
    'PC1_Loading': loadings_df['PC1_Loading'],
    'PC2_Loading': loadings_df['PC2_Loading']
})

print("=== Top 10 Positive Contributors to PC2 ===")
display(pc_summary['PC2_Loading'].sort_values(ascending=False).head(10))

print("\n=== Top 10 Negative Contributors to PC2 ===")
display(pc_summary['PC2_Loading'].sort_values(ascending=True).head(10))

## Analyze cluster visualizations

### Subtask:
Examine the generated plots to understand how well the clusters are separated and if the clusters with high default rates show distinct visual patterns.


## Summary:

### Data Analysis Key Findings

*   K-Means clustering was applied to the preprocessed data, resulting in 4 distinct customer clusters.
*   Visualizations using scatter plots of important features ('days\_elapsed\_negotiation' vs. 'borrower\_origination\_risk\_group' and 'days\_elapsed\_negotiation' vs. 'loan\_application\_NDI\_ratio'), a box plot ('days\_elapsed\_negotiation' by 'borrower\_origination\_risk\_group\_twentile'), and a stacked bar plot ('Cluster Distribution by Loan Vintage Quarter') were generated to understand the cluster characteristics.
*   PCA was applied to reduce the data to 2 dimensions, and a scatter plot of these principal components, colored by cluster label, was created to visualize cluster separation in a reduced space.

### Insights or Next Steps

*   Analyze the generated plots in detail to identify specific patterns, feature value ranges, and distributions that are characteristic of the clusters with high first pay default rates.
*   Quantify the separation between clusters using metrics beyond visual inspection, if necessary, to confirm the distinctiveness of the high-default clusters.


In [ ]:
from IPython.display import display, Markdown
import numpy as np

if 'cluster_assignment' not in globals():
    raise RuntimeError("Run Cluster alignment cell before summary.")
cluster_default_rates = df.groupby('cluster_label')['flag_1st_pay_default'].mean()

cluster_counts = df['cluster_label'].value_counts()
cluster_proportions = df['cluster_label'].value_counts(normalize=True) * 100
total_portfolio_loans = len(df)

summary_md = f"""## 🤖 Automated Cluster Profiles Summary (Total Portfolio Loans: {total_portfolio_loans:,})

"""

clusters = sorted(df['cluster_label'].unique())
for cluster_id in clusters:
    meta = cluster_assignment.get(cluster_id, {"name": f"Cluster {cluster_id}", "description": ""})
    def_rate = cluster_default_rates.loc[cluster_id] * 100
    size = cluster_counts.loc[cluster_id]
    pct = cluster_proportions.loc[cluster_id]
    rg_mean = numerical_characteristics_mean.loc[cluster_id, 'borrower_origination_risk_group']
    ndi_mean = numerical_characteristics_mean.loc[cluster_id, 'loan_application_NDI_ratio']
    pti_mean = numerical_characteristics_mean.loc[cluster_id, 'loan_application_PTI_ratio']
    var_mean = numerical_characteristics_mean.loc[cluster_id, 'fdr_vs_aal_payment_variance']
    amt_mean = numerical_characteristics_mean.loc[cluster_id, 'loan_origination_loan_amount']
    rate_mean = numerical_characteristics_mean.loc[cluster_id, 'loan_origination_interest_rate'] * 100
    home_mean = numerical_characteristics_mean.loc[cluster_id, 'borrower_home_owner_flag']
    neg_mean = numerical_characteristics_mean.loc[cluster_id, 'days_elapsed_negotiation']
    summary_md += f"""*   **{meta['name']}** ({meta.get('description', '')})
    *   **Segment Size:** {size:,} loans ({pct:.2f}% of total portfolio)
    *   **Default Rate:** {def_rate:.2f}%
    *   **Risk Profile:** Borrower Risk Group Mean: {rg_mean:.2f}
    *   **Financial Profile:** NDI: {ndi_mean:.2f}, PTI: {pti_mean:.2f} with an ADR vs. AAL Payment Variance of ${var_mean:.2f}
    *   **Loan Characteristics:** Average loan amount of ${amt_mean:,.2f} with a {rate_mean:.2f}% interest rate
    *   **Housing:** Homeownership rate ({home_mean:.2f})
    *   **Process:** Average negotiation period of {neg_mean:.2f} days

"""

display(Markdown(summary_md))


In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 1. Clean selected features using the master post_origination_cols from the clean split cell
# This ensures any new leakage columns added upstream are automatically dropped here too.
X_selected_clean = X_selected.drop(columns=post_origination_cols, errors='ignore')

# 2. Re-train the model with class weights to handle the highly imbalanced dataset
model_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model_balanced.fit(X_selected_clean, y)

# 3. Initialize SHAP Explainer
explainer = shap.TreeExplainer(model_balanced)
shap_values = explainer.shap_values(X_selected_clean)

# 4. Handle dimensions dynamically for multiclass output structures
if isinstance(shap_values, list):
    shap_to_plot = shap_values[1]
elif isinstance(shap_values, np.ndarray):
    if len(shap_values.shape) == 3:
        shap_to_plot = shap_values[:, :, 1]
    else:
        shap_to_plot = shap_values
else:
    shap_to_plot = shap_values

# 5. Generate the SHAP Summary Plot
plt.figure(figsize=(12, 8))
plt.title("SHAP Summary: Drivers of First Pay Default (Cleaned Pre-Origination Features Only)")
shap.summary_plot(shap_to_plot, X_selected_clean, show=False)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Post-origination diagnostics by cluster (labels from pre-origination K-Means only)
if 'df_profile' not in globals():
    df_profile = df.copy()
    df_profile['cluster_label_clean'] = df['cluster_label']

print("\n=== Diagnostic Return Code Profiling Across Clean Clusters ===")
return_code_profile = pd.crosstab(
    df_profile['cluster_label_clean'],
    df_profile['loan_1st_pay_default_return_code'].fillna('No Return Code'),
    normalize='index',
) * 100
display(return_code_profile.round(2))

print("\n=== Average Chargeoff & Delinquency Rates by Cluster ===")
profile_leakage_rates = df_profile.groupby('cluster_label_clean')[[
    'flag_1st_pay_default',
    'loan_chargeoff_flag',
    'loan_no_payment_straight_to_chargeoff_flag',
]].mean() * 100
display(profile_leakage_rates.round(2))


## Logistic Regression — Odds Ratios (Pre-Origination Features)

Odds ratios from a balanced logistic model on **pre-origination** selected features (`X_selected_clean`). Coefficients are on **standardized** inputs (1 SD increase). Pair with RF/SHAP for a credit-risk-friendly view.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Ensure clean feature matrix (no post-origination leakage)
if 'X_selected_clean' not in globals():
    X_selected_clean = X_selected.drop(columns=post_origination_cols, errors='ignore')

X_model = X_selected_clean.copy()
y_model = y.copy()
if len(X_model) != len(y_model):
    raise ValueError(f"Row mismatch: X={len(X_model)}, y={len(y_model)}")

scaler_lr = StandardScaler()
X_scaled = scaler_lr.fit_transform(X_model)

log_reg = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
log_reg.fit(X_scaled, y_model)

features = list(X_model.columns)
coefs = log_reg.coef_[0]
or_point = np.exp(coefs)

# Bootstrap 95% CI for odds ratios (per 1 SD)
rng = np.random.default_rng(42)
n_boot = 200
n = len(y_model)
boot_or = np.zeros((n_boot, len(features)))

for b in range(n_boot):
    idx = rng.integers(0, n, n)
    X_b = X_scaled[idx]
    y_b = y_model.iloc[idx] if hasattr(y_model, 'iloc') else y_model[idx]
    m = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    m.fit(X_b, y_b)
    boot_or[b, :] = np.exp(m.coef_[0])

or_low = np.percentile(boot_or, 2.5, axis=0)
or_high = np.percentile(boot_or, 97.5, axis=0)

or_df = pd.DataFrame({
    'Feature': features,
    'Odds_Ratio': or_point,
    'OR_CI_Low': or_low,
    'OR_CI_High': or_high,
    'Coefficient': coefs,
}).sort_values('Odds_Ratio', ascending=False)

print("--- Odds Ratios for First Pay Default (pre-origination features) ---")
print("Odds Ratio > 1: higher FPD odds per 1 SD increase (holding other features fixed).")
print("Odds Ratio < 1: lower FPD odds per 1 SD increase.\n")
display(or_df.round(3))


## Policy Rules — Highest-FPD Clean Cluster (Shallow Tree)

Transparent rules for the **clean** cluster with the highest mean `flag_1st_pay_default`, using only `X_selected_clean` (no post-origination leakage). K-Means cluster IDs are arbitrary — the target cluster is chosen by highest FPD rate, not a hard-coded label.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

if 'df_profile' not in globals() or 'cluster_label_clean' not in df_profile.columns:
    raise RuntimeError("Run the clean clustering / df_profile cell first.")

cluster_fpd = df_profile.groupby('cluster_label_clean')['flag_1st_pay_default'].mean()
critical_id = cluster_fpd.idxmax()
y_high_risk_cluster = (df_profile['cluster_label_clean'] == critical_id).astype(int)

n_pos = int(y_high_risk_cluster.sum())
n_total = len(y_high_risk_cluster)
print(f"Target: clean cluster {critical_id} | loans: {n_pos:,} / {n_total:,} "
      f"({100*n_pos/n_total:.2f}%) | mean FPD: {cluster_fpd.loc[critical_id]*100:.2f}%")

X_tree = X_selected_clean.copy()
if len(X_tree) != len(y_high_risk_cluster):
    X_tree = X_tree.loc[df_profile.index]

tree_rules = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
tree_rules.fit(X_tree, y_high_risk_cluster)

print("\n--- Underwriting-style rules (max depth 3) to flag highest-FPD clean cluster ---")
print(export_text(tree_rules, feature_names=list(X_tree.columns)))

imp = pd.Series(tree_rules.feature_importances_, index=X_tree.columns).sort_values(ascending=False)
print("\nTree feature importance (for this cluster-membership target):")
display(imp[imp > 0].head(10).round(4))
